# Análisis de sentimiento de reseñas

Comparación reproducible de tres clasificadores binarios en reseñas en inglés. Se evaluarán accuracy, precision, recall y F1, además de latencia de inferencia en el entorno actual. Las métricas que aparezcan en las tablas se calculan al ejecutar el cuaderno; no se presentan resultados inventados.

## 1. Configuración del entorno


In [3]:
!pip -q install -U "transformers>=4.44,<5" "datasets>=2.20" "evaluate>=0.4" "accelerate>=0.33" "kagglehub>=0.3" "scikit-learn>=1.4" pandas beautifulsoup4

In [4]:
import os, time, glob, zipfile, subprocess, html, re, warnings
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
from transformers import pipeline
from huggingface_hub import notebook_login, login
from google.colab import files, userdata

SEED = 42
EVAL_LIMIT = 2000  # Usa None para evaluar las 50 000 reseñas; requiere más tiempo.
BATCH_SIZE = 16
MAX_LENGTH = 512
DEVICE = 0 if torch.cuda.is_available() else -1
print('PyTorch:', torch.__version__, '| GPU disponible:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


PyTorch: 2.11.0+cu130 | GPU disponible: True
Tesla T4


## 2. Autenticación de Hugging Face Hub


In [5]:
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token) if hf_token else notebook_login()
except Exception as e:
    print('No se inició sesión (opcional para estos modelos públicos):', type(e).__name__)

No se inició sesión (opcional para estos modelos públicos): SecretNotFoundError


## 3. Descarga del dataset de Kaggle

Fuente: [IMDB Dataset of 50K Movie Reviews](https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews). El archivo fuente contiene `review` (texto) y `sentiment` (etiqueta `positive`/`negative`).

In [6]:
import kagglehub

# Descarga la versión más reciente y usa el directorio local que devuelve KaggleHub.
path = kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews")
print("Path to dataset files:", path)
csvs = glob.glob(os.path.join(path, '**', '*.csv'), recursive=True)
if not csvs:
    raise FileNotFoundError(f'No se encontró el CSV descargado en {path}')
print(csvs)

Using Colab cache for faster access to the 'imdb-dataset-of-50k-movie-reviews' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-50k-movie-reviews
['/kaggle/input/imdb-dataset-of-50k-movie-reviews/IMDB Dataset.csv']


In [7]:
df = pd.read_csv(csvs[0])
# Normaliza nombres de columnas sin alterar el contenido semántico.
df.columns = [c.strip().lower() for c in df.columns]
assert {'review','sentiment'}.issubset(df.columns), f'Columnas inesperadas: {df.columns.tolist()}'
df = df[['review','sentiment']].dropna().drop_duplicates().copy()
df['label'] = df['sentiment'].str.lower().map({'negative':0,'positive':1})
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
print('Filas válidas:', len(df))
display(df['sentiment'].value_counts().rename_axis('clase').to_frame('n'))
display(df.head(3))

Filas válidas: 49582


,n
clase,
positive,24884
negative,24698


,review,sentiment,label
0,One of the other reviewers has mentioned that ...,positive,1
1,A wonderful little production. <br /><br />The...,positive,1
2,I thought this was a wonderful way to spend ti...,positive,1


## 4. División estratificada

El archivo de Kaggle no trae un split explícito en esta copia; se crea una partición reproducible 80/20, manteniendo proporción de clases. Se mide sobre test y no se ajustan pesos. La muestra de ejecución predeterminada se toma de test; fija `EVAL_LIMIT=None` para usar todo test.

In [8]:
train_df, test_df = train_test_split(df, test_size=0.20, random_state=SEED, stratify=df['label'])
if EVAL_LIMIT is not None and EVAL_LIMIT < len(test_df):
    _, eval_df = train_test_split(test_df, test_size=EVAL_LIMIT, random_state=SEED, stratify=test_df['label'])
else:
    eval_df = test_df.copy()
texts = eval_df['review'].astype(str).tolist()
y_true = eval_df['label'].astype(int).to_numpy()
print(f'Test disponible: {len(test_df):,}; evaluado: {len(eval_df):,}; positivos: {y_true.mean():.3f}')

Test disponible: 9,917; evaluado: 2,000; positivos: 0.502


## 5. Modelos y pipeline de inferencia

Los tres modelos realizan clasificación binaria en inglés. El tercero es un RoBERTa grande con entrenamiento diverso; sus requisitos son mayores. Las salidas se convierten a `0=negativo, 1=positivo` a partir de `id2label`, con inspección de etiquetas.

In [9]:
MODEL_IDS = {
    'DistilBERT SST-2': 'distilbert/distilbert-base-uncased-finetuned-sst-2-english',
    'BERT SST-2': 'textattack/bert-base-uncased-SST-2',
    'RoBERTa large SiEBERT': 'siebert/sentiment-roberta-large-english',
}

def map_label(label):
    s = str(label).lower()
    if 'positive' in s or s in ('label_1','1'): return 1
    if 'negative' in s or s in ('label_0','0'): return 0
    raise ValueError(f'Etiqueta no reconocida: {label}')

def sync_device():
    if torch.cuda.is_available(): torch.cuda.synchronize()

def evaluate_model(name, model_id):
    t0=time.perf_counter()
    clf=pipeline('text-classification', model=model_id, tokenizer=model_id,
                 device=DEVICE, truncation=True, max_length=MAX_LENGTH)
    load_s=time.perf_counter()-t0
    print(name, 'labels:', clf.model.config.id2label)
    # Warm-up excluido de la latencia reportada.
    _=clf(texts[:min(8,len(texts))], batch_size=min(8,len(texts)))
    sync_device(); t0=time.perf_counter()
    preds=clf(texts, batch_size=BATCH_SIZE)
    sync_device(); total_s=time.perf_counter()-t0
    y_pred=np.array([map_label(p['label']) for p in preds])
    row={'modelo':name,'accuracy':accuracy_score(y_true,y_pred),
         'precision':precision_score(y_true,y_pred,zero_division=0),
         'recall':recall_score(y_true,y_pred,zero_division=0),
         'f1':f1_score(y_true,y_pred,zero_division=0),
         'latencia_ms_por_reseña':1000*total_s/len(texts),
         'reseñas_por_segundo':len(texts)/total_s,'carga_s':load_s}
    return row, y_pred

rows=[]; predictions={}
for name, model_id in MODEL_IDS.items():
    row, pred=evaluate_model(name, model_id)
    rows.append(row); predictions[name]=pred
    print(classification_report(y_true,pred,target_names=['negativa','positiva'],zero_division=0))
results=pd.DataFrame(rows).sort_values(['f1','latencia_ms_por_reseña'],ascending=[False,True]).reset_index(drop=True)
display(results.style.format({c:'{:.4f}' for c in results.columns if c not in ['modelo']}))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Device set to use cuda:0


... (15 outputs left out)

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

Device set to use cuda:0


RoBERTa large SiEBERT labels: {0: 'NEGATIVE', 1: 'POSITIVE'}
              precision    recall  f1-score   support

    negativa       0.96      0.95      0.96       996
    positiva       0.95      0.96      0.96      1004

    accuracy                           0.96      2000
   macro avg       0.96      0.96      0.96      2000
weighted avg       0.96      0.96      0.96      2000



,modelo,accuracy,precision,recall,f1,latencia_ms_por_reseña,reseñas_por_segundo,carga_s
0,RoBERTa large SiEBERT,0.9570,0.9509,0.9641,0.9575,94.9640,10.5303,31.5761
1,BERT SST-2,0.9055,0.9137,0.8964,0.9050,32.9384,30.3597,15.0558
2,DistilBERT SST-2,0.8870,0.9147,0.8546,0.8836,16.2070,61.7018,8.3623


## 6. Medición de latencia de una reseña

La tabla principal mide rendimiento por lote e informa latencia amortizada por reseña y throughput. La siguiente celda estima además latencia individual secuencial tras calentamiento.

In [11]:
def single_item_latency(model_id, sample_text, repeats=30):
    clf=pipeline('text-classification', model=model_id, tokenizer=model_id,
                 device=DEVICE, truncation=True, max_length=MAX_LENGTH)
    for _ in range(3): clf(sample_text)
    timings=[]
    for _ in range(repeats):
        sync_device(); t=time.perf_counter(); clf(sample_text); sync_device()
        timings.append(1000*(time.perf_counter()-t))
    return float(np.median(timings)), float(np.percentile(timings,95))
lat=[]
for name, mid in MODEL_IDS.items():
    med,p95=single_item_latency(mid, texts[0])
    lat.append({'modelo':name,'latencia_individual_mediana_ms':med,'latencia_individual_p95_ms':p95})
latency_df=pd.DataFrame(lat)
display(latency_df.round(2))

Device set to use cuda:0
Device set to use cuda:0
Device set to use cuda:0


,modelo,latencia_individual_mediana_ms,latencia_individual_p95_ms
0,DistilBERT SST-2,10.92,12.41
1,BERT SST-2,13.94,18.26
2,RoBERTa large SiEBERT,45.51,46.70


## 7. Resumen, umbrales y conclusiones parciales

Criterio de aula sugerido (ajustable): F1 ≥ 0.85 y latencia mediana individual ≤ 250 ms en el hardware de ejecución. Para una aplicación real, define el umbral con usuarios y costos observados. Accuracy/F1 en reseñas equilibradas no garantizan desempeño en reseñas cortas, sarcásticas, multilingües ni en otras plataformas.

In [12]:
F1_MIN=0.85
LATENCY_MAX_MS=250
summary=results.merge(latency_df,on='modelo')
summary['cumple_criterios']=(summary['f1']>=F1_MIN)&(summary['latencia_individual_mediana_ms']<=LATENCY_MAX_MS)
display(summary[['modelo','accuracy','precision','recall','f1','latencia_ms_por_reseña','latencia_individual_mediana_ms','latencia_individual_p95_ms','cumple_criterios']].round(4))
eligible=summary[summary['cumple_criterios']].sort_values(['f1','latencia_individual_mediana_ms'],ascending=[False,True])
if len(eligible):
    best=eligible.iloc[0]
    print(f"Candidato según umbral actual: {best['modelo']} (F1={best['f1']:.3f}, mediana={best['latencia_individual_mediana_ms']:.1f} ms).")
else:
    print('Ningún modelo cumple simultáneamente los umbrales actuales; revisa latencia, hardware o umbrales.')
print('Conclusión parcial: prioriza F1 si los errores de ambas clases importan; compara latencia individual para aplicaciones interactivas y throughput para procesamiento por lotes.')

,modelo,accuracy,precision,recall,f1,latencia_ms_por_reseña,latencia_individual_mediana_ms,latencia_individual_p95_ms,cumple_criterios
0,RoBERTa large SiEBERT,0.9570,0.9509,0.9641,0.9575,94.9640,45.5068,46.7021,True
1,BERT SST-2,0.9055,0.9137,0.8964,0.9050,32.9384,13.9373,18.2571,True
2,DistilBERT SST-2,0.8870,0.9147,0.8546,0.8836,16.2070,10.9165,12.4061,True


Candidato según umbral actual: RoBERTa large SiEBERT (F1=0.957, mediana=45.5 ms).
Conclusión parcial: prioriza F1 si los errores de ambas clases importan; compara latencia individual para aplicaciones interactivas y throughput para procesamiento por lotes.


## 8. Exportación de resultados


In [ ]:
results.to_csv('/content/metricas_modelos.csv',index=False)
latency_df.to_csv('/content/latencias_individuales.csv',index=False)
print('/content/metricas_modelos.csv y /content/latencias_individuales.csv')


/content/metricas_modelos.csv y /content/latencias_individuales.csv
